# CAFA 6 - ESM-2 Embedding Extraction

Extract protein embeddings using ESM-2 (Facebook's protein language model).

**GPU Required**: This notebook needs a GPU. Options:
- Google Colab Pro+ (A100) - recommended for notebooks
- Kaggle (free P100, 30hrs/week)
- SSH into Lambda Labs / RunPod (fastest)

**Model choices:**
| Model | Params | Embed Dim | VRAM | ~Time (82K train seqs) |
|-------|--------|-----------|------|------------------------|
| esm2_t12_35M | 35M | 480 | ~2GB | ~30min (T4) |
| esm2_t33_650M | 650M | 1280 | ~8GB | ~3hr (T4) |
| esm2_t36_3B | 3B | 2560 | ~24GB | ~12hr (A100) |

In [ ]:
# === Google Colab Setup (skip if running locally) ===
import os
if 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ:
    !git clone https://github.com/AyushSharma173/CafaChallenge.git
    %cd CafaChallenge
    !pip install -q obonet biopython h5py lightgbm scikit-learn matplotlib seaborn pyyaml
    !pip install -q fair-esm torch
    !pip install -q kaggle
    os.makedirs('/root/.kaggle', exist_ok=True)
    if os.path.exists('kaggle.json'):
        !cp kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
    !kaggle competitions download -c cafa-6-protein-function-prediction -p data/raw
    !unzip -qo data/raw/cafa-6-protein-function-prediction.zip -d data/raw/
    %cd notebooks
    print('Colab setup complete!')

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import numpy as np
from pathlib import Path

from cafa6.data_loader import load_fasta
from cafa6.embeddings import load_esm_model, extract_embeddings, load_embeddings

# Auto-detect device
if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('Using Apple MPS')
else:
    DEVICE = 'cpu'
    print('WARNING: No GPU detected. This will be very slow.')

DATA_DIR = Path('../data/raw')
EMB_DIR = Path('../data/embeddings')
EMB_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Model

In [ ]:
MODEL_NAME = 'esm2_t33_650M_UR50D'  # Best cost/performance
REPR_LAYER = 33
BATCH_SIZE = 4  # Increase for more VRAM (8 for A100, 2 for T4)

model, alphabet, batch_converter = load_esm_model(MODEL_NAME, DEVICE)
print(f'Model loaded: {MODEL_NAME}')

## 2. Quick Test (10 proteins)

In [ ]:
train_seqs = load_fasta(DATA_DIR / 'Train' / 'train_sequences.fasta')
print(f'Loaded {len(train_seqs)} training sequences')

# Test with 10 proteins
test_subset = dict(list(train_seqs.items())[:10])
extract_embeddings(
    sequences=test_subset,
    model=model, alphabet=alphabet, batch_converter=batch_converter,
    repr_layer=REPR_LAYER, batch_size=2, device=DEVICE,
    output_path=str(EMB_DIR / 'test_quick.h5')
)

# Verify
embs, ids = load_embeddings(EMB_DIR / 'test_quick.h5')
print(f'Embeddings shape: {embs.shape}')  # Should be (10, 1280)
print(f'Protein IDs: {ids[:3]}...')

## 3. Full Training Set Extraction

This takes ~2-6 hours depending on GPU. The pipeline checkpoints every 1000 proteins.

In [ ]:
# Full training set
extract_embeddings(
    sequences=train_seqs,
    model=model, alphabet=alphabet, batch_converter=batch_converter,
    repr_layer=REPR_LAYER, batch_size=BATCH_SIZE, device=DEVICE,
    output_path=str(EMB_DIR / f'train_{MODEL_NAME}.h5'),
    checkpoint_every=1000
)

In [ ]:
# Full test set
test_seqs = load_fasta(DATA_DIR / 'Test' / 'testsuperset.fasta')
print(f'Loaded {len(test_seqs)} test sequences')

extract_embeddings(
    sequences=test_seqs,
    model=model, alphabet=alphabet, batch_converter=batch_converter,
    repr_layer=REPR_LAYER, batch_size=BATCH_SIZE, device=DEVICE,
    output_path=str(EMB_DIR / f'test_{MODEL_NAME}.h5'),
    checkpoint_every=1000
)

## 4. Verify & Visualize Embeddings

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

embs, ids = load_embeddings(EMB_DIR / f'train_{MODEL_NAME}.h5')
print(f'Train embeddings: {embs.shape}')

# PCA visualization (subsample for speed)
n_sample = min(5000, len(embs))
idx = np.random.choice(len(embs), n_sample, replace=False)
embs_sample = embs[idx].astype(np.float32)

pca = PCA(n_components=2)
coords = pca.fit_transform(embs_sample)

fig, ax = plt.subplots(figsize=(10, 8))
ax.scatter(coords[:, 0], coords[:, 1], alpha=0.3, s=5)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
ax.set_title(f'ESM-2 Embeddings PCA ({MODEL_NAME})')
plt.tight_layout()
plt.show()